# build_knowledge_colab.py

In [ ]:
%%writefile build_knowledge_colab.py
import os
import json
from openai import OpenAI
import tempfile


def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_json(path, data):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def normalize_files_cache(files_cache):
    if isinstance(files_cache, dict):
        return files_cache
    return {}


def build_knowledge(
    api_key,
    knowledge_files,
    vector_store_name,
    cache_path,
    force_new_vector_store=False,
):
    client = OpenAI(api_key=api_key)

    cache = load_json(cache_path)
    files_cache = normalize_files_cache(cache.get("files", {}))
    vector_store_id = None if force_new_vector_store else cache.get("vector_store", {}).get("id")

    uploaded_file_ids = []
    files_to_process = [] # List of (original_filepath, filename_for_cache)

    temp_files_created = []

    for filepath in knowledge_files:
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")

        print(f"DEBUG (loop 1): Processing filepath: '{filepath}'")
        print(f"DEBUG (loop 1): filepath.lower(): '{filepath.lower()}'")
        print(f"DEBUG (loop 1): filepath.lower().endswith('.jsonl'): {filepath.lower().endswith('.jsonl')}")

        # Handle .jsonl files specifically by converting them to .txt
        if filepath.lower().endswith(".jsonl"):
            temp_txt_file_path = None
            try:
                # Create a temporary .txt file to upload
                base_filename = os.path.basename(filepath)
                # tempfile.NamedTemporaryFile creates a unique file, let's just make one in a temp directory
                temp_dir = tempfile.gettempdir()
                temp_txt_file_path = os.path.join(temp_dir, f"temp_{os.path.splitext(base_filename)[0]}.txt")

                print(f"DEBUG: base_filename: {base_filename}")
                print(f"DEBUG: temp_txt_file_path: {temp_txt_file_path}")
                print(f"DEBUG: filename_for_cache from temp path: {os.path.basename(temp_txt_file_path)}")

                print(f"Converting {filepath} to temporary text file: {temp_txt_file_path}")
                with open(temp_txt_file_path, mode='w', encoding='utf-8') as temp_f:
                    with open(filepath, 'r', encoding='utf-8') as jsonl_f:
                        for line_num, line in enumerate(jsonl_f, 1):
                            try:
                                # Assuming each line is a JSON object. Flatten it to a string.
                                json_obj = json.loads(line)
                                # If json_obj is a dict, you might want to format its contents more explicitly
                                # For now, a simple string representation is sufficient for a knowledge base
                                temp_f.write(json.dumps(json_obj, ensure_ascii=False) + "\n")
                            except json.JSONDecodeError as e:
                                print(f"Warning: Skipping invalid JSON line {line_num} in {filepath}: {e}")
                # Use the basename of the temporary file for filename_for_cache
                files_to_process.append((temp_txt_file_path, os.path.basename(temp_txt_file_path)))
                temp_files_created.append(temp_txt_file_path)

            except Exception as e:
                if temp_txt_file_path and os.path.exists(temp_txt_file_path):
                    os.remove(temp_txt_file_path)
                raise RuntimeError(f"Error processing .jsonl file {filepath}: {e}")
        else:
            # For other file types, use the original path and filename
            # This branch should NOT be taken for .jsonl files.
            print(f"DEBUG (loop 1 else): Processing non-.jsonl file: {filepath}")
            print(f"DEBUG (loop 1 else): filename_for_cache: {os.path.basename(filepath)}")
            files_to_process.append((filepath, os.path.basename(filepath)))

    for filepath_to_upload, filename_for_cache in files_to_process:
        print(f"DEBUG (loop 2): filepath_to_upload: '{filepath_to_upload}'")
        print(f"DEBUG (loop 2): filename_for_cache (used for cache key and display): '{filename_for_cache}'")
        if filename_for_cache in files_cache and not force_new_vector_store:
            file_id = files_cache[filename_for_cache]
            print(f"Reuse file: {filename_for_cache} -> {file_id}")
        else:
            print(f"Uploading: {filename_for_cache}")
            with open(filepath_to_upload, "rb") as f:
                # Explicitly provide filename to OpenAI to prevent misinterpretation
                uploaded = client.files.create(file=(filename_for_cache, f.read()), purpose="assistants")
            file_id = uploaded.id
            files_cache[filename_for_cache] = file_id
            print(f"Uploaded: {filename_for_cache} -> {file_id}")

        uploaded_file_ids.append(file_id)

    if vector_store_id and not force_new_vector_store:
        print(f"Reuse vector store: {vector_store_id}")
    else:
        print("Creating vector store...")
        vs = client.vector_stores.create(name=vector_store_name)
        vector_store_id = vs.id
        print(f"Created vector store: {vector_store_id}")

    print("Attaching files and waiting for indexing to complete...")
    batch = client.vector_stores.file_batches.create_and_poll(
        vector_store_id=vector_store_id,
        file_ids=uploaded_file_ids
    )

    print(f"Batch status: {batch.status}")
    if getattr(batch, "file_counts", None):
        print(f"File counts: {batch.file_counts}")

    if batch.status != "completed":
        raise RuntimeError(f"Vector store ingestion not completed. Status={{batch.status}}")

    cache = {
        "files": files_cache,
        "vector_store": {
            "id": vector_store_id
        }
    }
    save_json(cache_path, cache)

    print("\nDONE")
    print("vector_store_id:", vector_store_id)
    print("cache_path:", cache_path)

    # Clean up temporary files
    for temp_path in temp_files_created:
        try:
            os.remove(temp_path)
            print(f"Cleaned up temporary file: {temp_path}")
        except OSError as e:
            print(f"Error cleaning up temporary file {temp_path}: {e}")

    return {
        "vector_store_id": vector_store_id,
        "cache_path": cache_path,
        "batch_status": batch.status,
    }

Writing build_knowledge_colab.py


# run_inference_colab.py

In [ ]:
%%writefile run_inference_colab.py
import os
import json
import time
import re
import traceback
from typing import List, Dict, Any

import pandas as pd
from pydantic import BaseModel
from openai import OpenAI, APIConnectionError, APITimeoutError, RateLimitError, APIError


class MediCheckOutput(BaseModel):
    order_id: str
    has_medication_error: bool
    error_categories: List[str]
    error_details: List[str]
    implicated_drugs: List[str]


def load_json(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def save_json(path: str, data) -> None:
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def append_debug_log(path: str, payload: Dict[str, Any]) -> None:
    logs = load_json(path, [])
    logs.append(payload)
    save_json(path, logs)


def normalize_order_id(value: Any) -> str:
    if value is None:
        return "UNKNOWN"
    return str(value).strip()


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(how="all")
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    df = df.where(pd.notnull(df), None)
    df = df.drop_duplicates()

    if "Drug" in df.columns:
        df = df[~df["Drug"].astype(str).str.strip().isin(["", "-", "None", "nan"])]

    return df


def dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    result = []
    for item in items:
        if item is None:
            continue
        item = str(item).strip()
        if not item:
            continue
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


def get_instructions() -> str:
    return """
You are a clinical AI system for classifying prescription medication errors.
Core task:
- Identify and classify medication errors using ONLY the provided prescription data and retrieved knowledge excerpts.
Strict rules:
1. Evidence-based only:
   - Every detected medication error MUST be supported by explicit evidence from the provided knowledge.
   - Do NOT infer, assume, or use external knowledge.
2. Insufficient evidence:
   - If evidence is missing, unclear, or not directly supported → set "has_medication_error" to false.
3. Multi-label classification:
   - Multiple error types can be assigned if independently supported by evidence.
4. No hallucination:
   - Do NOT generate explanations beyond the given knowledge.
   - Do NOT guess clinical relationships.
5. Precision over recall:
   - It is better to miss an error than to falsely label one.
Output format:
- Return JSON only.
- No extra text.
""".strip()


def get_json_schema() -> Dict[str, Any]:
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "order_id": {"type": "string"},
            "has_medication_error": {"type": "boolean"},
            "error_categories": {
                "type": "array",
                "items": {"type": "string"}
            },
            "error_details": {
                "type": "array",
                "items": {"type": "string"}
            },
            "implicated_drugs": {
                "type": "array",
                "items": {"type": "string"}
            }
        },
        "required": [
            "order_id",
            "has_medication_error",
            "error_categories",
            "error_details",
            "implicated_drugs"
        ]
    }


def build_user_prompt(order_id: str, prescription_records: List[Dict[str, Any]]) -> str:
    compact_records = []
    for row in prescription_records:
        compact_records.append({
            "Drug": row.get("Drug"),
            "Dose": row.get("Dose"),
            "Frequency": row.get("Frequency"),
        })

    prescription_text = json.dumps(compact_records, ensure_ascii=False, separators=(",", ":"))

    return (
        f"Order ID: {order_id}\n"
        f"Prescription: {prescription_text}\n"
        f"Task: classify medication error(s) with multi-label output.\n"
        f"Return JSON only with keys: order_id, has_medication_error, error_categories, error_details, implicated_drugs"
    )


def build_kb_query(prescription_records: List[Dict[str, Any]]) -> str:
    drugs = []
    for row in prescription_records:
        drug = row.get("Drug")
        if drug:
            d = str(drug).strip()
            if d and d not in ["-", "None", "nan"]:
                drugs.append(d)

    unique_drugs = dedupe_preserve_order(drugs)
    drug_part = "; ".join(unique_drugs[:5])

    return f"medication error classification taxonomy relevant drugs {drug_part}".strip()


def retrieve_kb_snippets(
    client: OpenAI,
    vector_store_id: str,
    query: str,
    max_results: int = 3
) -> str:
    result = client.vector_stores.search(
        vector_store_id=vector_store_id,
        query=query,
        max_num_results=max_results,
    )

    snippets = []
    data = getattr(result, "data", []) or []

    for item in data:
        content = getattr(item, "content", []) or []
        parts = []

        for c in content:
            text_value = getattr(c, "text", None)
            if text_value:
                parts.append(text_value)

        joined = "\n".join(parts).strip()
        if joined:
            snippets.append(joined)

    return "\n\n---\n\n".join(snippets[:max_results])


def extract_json_from_text(text: str) -> Dict[str, Any]:
    if text is None:
        raise ValueError("response text is None")

    text = text.strip()
    if not text:
        raise ValueError("response text is empty")

    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError(f"Could not parse valid JSON from model output: {repr(text[:1000])}")


def call_model(
    client: OpenAI,
    model: str,
    vector_store_id: str,
    instructions: str,
    user_prompt: str,
    prescription_records: List[Dict[str, Any]],
    max_num_results: int = 3,
    debug: bool = False,
    max_retries: int = 3,
) -> Dict[str, Any]:
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            kb_query = build_kb_query(prescription_records)
            kb_snippets = retrieve_kb_snippets(
                client=client,
                vector_store_id=vector_store_id,
                query=kb_query,
                max_results=max_num_results,
            )

            if not kb_snippets.strip():
                raise ValueError("No knowledge snippets were retrieved from the vector store.")

            combined_input = f"KB:{kb_snippets}\nCASE:{user_prompt}"

            response = client.responses.create(
                model=model,
                instructions=instructions,
                input=combined_input,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "medicheck_output",
                        "strict": True,
                        "schema": get_json_schema()
                    }
                }
            )

            raw_text = getattr(response, "output_text", None)

            if debug:
                print("----- KB QUERY -----")
                print(kb_query[:500])
                print("----- KB SNIPPETS -----")
                print(kb_snippets[:1500])
                print("----- RESPONSE PREVIEW -----")
                try:
                    print(str(response.model_dump())[:2000])
                except Exception:
                    print(str(response)[:2000])

            if not raw_text or not raw_text.strip():
                preview = response.model_dump() if hasattr(response, "model_dump") else str(response)
                raise ValueError(f"Empty final model output. Response preview: {str(preview)[:2000]}")

            if debug:
                print("----- RAW MODEL TEXT -----")
                print(repr(raw_text[:1500]))
                print("----- END RAW MODEL TEXT -----")

            return extract_json_from_text(raw_text)

        except (APIConnectionError, APITimeoutError, RateLimitError, APIError, ConnectionError) as e:
            last_error = e
            wait_sec = min(2 ** attempt, 10)
            print(f"API/network error attempt {attempt}/{max_retries}: {e}")
            time.sleep(wait_sec)

        except Exception as e:
            last_error = e
            wait_sec = min(2 ** attempt, 10)
            print(f"Error attempt {attempt}/{max_retries}: {e}")
            time.sleep(wait_sec)

    raise last_error


def postprocess_output(parsed: Dict[str, Any], order_id: str) -> Dict[str, Any]:
    parsed["order_id"] = normalize_order_id(parsed.get("order_id", order_id))

    for key in [
        "error_categories",
        "error_details",
        "implicated_drugs",
    ]:
        parsed[key] = dedupe_preserve_order(parsed.get(key, []))

    if not parsed.get("has_medication_error", False):
        parsed["error_categories"] = []
        parsed["error_details"] = []
        parsed["implicated_drugs"] = []

    validated = MediCheckOutput(**parsed)
    return validated.model_dump()


def run_inference(
    api_key,
    excel_file,
    group_column,
    cache_path,
    output_dir,
    model="gpt-4o-mini",
    sleep_sec=1.0,
    autosave_every=10,
    max_num_results=3,
    debug=True,
):
    client = OpenAI(api_key=api_key)

    cache = load_json(cache_path, {})
    vector_store_id = cache.get("vector_store", {}).get("id")
    if not vector_store_id:
        raise ValueError("vector_store_id not found in cache")

    os.makedirs(output_dir, exist_ok=True)

    partial_file = os.path.join(output_dir, "partial_results.json")
    failed_file = os.path.join(output_dir, "failed_cases.json")
    final_file = os.path.join(output_dir, "medicheck_results.json")
    debug_file = os.path.join(output_dir, "debug_log.json")

    results: Dict[str, Any] = load_json(partial_file, {})
    failed_cases: List[Dict[str, Any]] = load_json(failed_file, [])

    if not os.path.exists(excel_file):
        raise FileNotFoundError(f"Input file not found: {excel_file}")

    if excel_file.lower().endswith(".json"):
        with open(excel_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    else:
        df = pd.read_excel(excel_file)

    df = clean_dataframe(df)

    if group_column not in df.columns:
        raise ValueError(f"Column '{group_column}' not found. Available columns: {list(df.columns)}")

    grouped = list(df.groupby(group_column, dropna=False))
    total_cases = len(grouped)

    remaining = []
    already_processed = 0

    for order_id, group_df in grouped:
        oid = normalize_order_id(order_id)
        if oid in results:
            already_processed += 1
        else:
            remaining.append((oid, group_df))

    print(f"Already processed: {already_processed}")
    print(f"Remaining: {len(remaining)} / {total_cases}")

    instructions = get_instructions()

    for idx, (order_id, group_df) in enumerate(remaining, start=1):
        current_case = already_processed + idx
        print(f"\nProcessing {current_case}/{total_cases} -> Order {order_id}")

        user_prompt = None

        try:
            prescription_records = group_df.to_dict(orient="records")
            user_prompt = build_user_prompt(order_id, prescription_records)

            parsed = call_model(
                client=client,
                model=model,
                vector_store_id=vector_store_id,
                instructions=instructions,
                user_prompt=user_prompt,
                prescription_records=prescription_records,
                max_num_results=max_num_results,
                debug=debug,
            )

            final_output = postprocess_output(parsed, order_id)
            results[order_id] = final_output
            print(f"Saved result for {order_id}")

        except Exception as e:
            failed_cases.append({
                "order_id": order_id,
                "error_type": type(e).__name__,
                "error": str(e),
                "traceback": traceback.format_exc()
            })
            append_debug_log(debug_file, {
                "order_id": order_id,
                "error_type": type(e).__name__,
                "error": str(e),
                "traceback": traceback.format_exc(),
                "prompt_preview": user_prompt[:2000] if user_prompt else None
            })
            print(f"Error on {order_id}: {e}")

        time.sleep(sleep_sec)

        if current_case % autosave_every == 0:
            save_json(partial_file, results)
            save_json(failed_file, failed_cases)
            print(f"Autosaved after {current_case} cases")

    save_json(final_file, results)
    save_json(failed_file, failed_cases)

    print("\nDONE")
    print(f"Total processed: {len(results)}")
    print(f"Failed: {len(failed_cases)}")
    print(f"Results saved to: {final_file}")
    print(f"Debug log saved to: {debug_file}")

    return {
        "results_file": final_file,
        "failed_file": failed_file,
        "debug_file": debug_file,
        "processed": len(results),
        "failed": len(failed_cases),
    }

Writing run_inference_colab.py


# Variations

In [ ]:
import os
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive')

OPENAI_API_KEY = userdata.get("GPTKEY")

KNOWLEDGE_FILES = [

    "/content/drive/MyDrive/thesis/KB/medication_error_kb_chunked.jsonl",
    "/content/drive/MyDrive/thesis/KB/BKN_Med_List.jsonl",
]

VECTOR_STORE_NAME = "MediCheck Medication Error KB"
CACHE_PATH = "/content/drive/MyDrive/thesis/Result/medicheck_cache/knowledge_cache.json"

EXCEL_FILE = "/content/drive/MyDrive/thesis/RT_COMMON_904_test_clean_blinded_first_sheet.json"
GROUP_COLUMN = "ID"
OUTPUT_DIR = "/content/drive/MyDrive/thesis/Result/thesismedicheck_output"

MODEL = "gpt-4o-mini"
DEBUG = True

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Create knowledge base

In [ ]:
from build_knowledge_colab import build_knowledge
import importlib
import build_knowledge_colab

importlib.reload(build_knowledge_colab)

# ✅ ensure file paths exist
for f in KNOWLEDGE_FILES:
    print(f, "->", "OK" if os.path.exists(f) else "NOT FOUND")

build_result = build_knowledge(
    api_key=OPENAI_API_KEY,

    # ✅ รองรับทั้ง PDF + JSON + JSONL
    knowledge_files=KNOWLEDGE_FILES,

    vector_store_name=VECTOR_STORE_NAME,

    # ✅ cache สำคัญมาก
    cache_path=CACHE_PATH,

    # ⚠️ ใช้ True เฉพาะตอน rebuild KB
    force_new_vector_store=True,

    # ✅ NEW (สำคัญ)
    # chunk_json=True,          # split JSON → chunk อัตโนมัติ
    # jsonl_mode=True,          # ถ้าเป็น .jsonl → ไม่ต้อง parseซ้ำ

    # ✅ ปรับ performance
    # max_chunk_size=800,       # ลด token
    # overlap=100               # กัน context ขาด
)

build_result

/content/drive/MyDrive/thesis/KB/medication_error_kb_chunked.jsonl -> OK
/content/drive/MyDrive/thesis/KB/BKN_Med_List.jsonl -> OK
DEBUG (loop 1): Processing filepath: '/content/drive/MyDrive/thesis/KB/medication_error_kb_chunked.jsonl'
DEBUG (loop 1): filepath.lower(): '/content/drive/mydrive/thesis/kb/medication_error_kb_chunked.jsonl'
DEBUG (loop 1): filepath.lower().endswith('.jsonl'): True
DEBUG: base_filename: medication_error_kb_chunked.jsonl
DEBUG: temp_txt_file_path: /tmp/temp_medication_error_kb_chunked.txt
DEBUG: filename_for_cache from temp path: temp_medication_error_kb_chunked.txt
Converting /content/drive/MyDrive/thesis/KB/medication_error_kb_chunked.jsonl to temporary text file: /tmp/temp_medication_error_kb_chunked.txt
DEBUG (loop 1): Processing filepath: '/content/drive/MyDrive/thesis/KB/BKN_Med_List.jsonl'
DEBUG (loop 1): filepath.lower(): '/content/drive/mydrive/thesis/kb/bkn_med_list.jsonl'
DEBUG (loop 1): filepath.lower().endswith('.jsonl'): True
DEBUG: base_filen

{'vector_store_id': 'vs_69be3dfe249881919d287249bebbcd2e',
 'cache_path': '/content/drive/MyDrive/thesis/Result/medicheck_cache/knowledge_cache.json',
 'batch_status': 'completed'}

# Run Inference

In [ ]:
from run_inference_colab import run_inference

run_result = run_inference(
    api_key=OPENAI_API_KEY,

    #  JSON / JSONL แล้ว
    excel_file=EXCEL_FILE,

    group_column=GROUP_COLUMN,

    cache_path=CACHE_PATH,
    output_dir=OUTPUT_DIR,

    model=MODEL,

    sleep_sec=1.0,
    autosave_every=10,

    # 🔥 ลด token + เร็วขึ้น
    max_num_results=4,

    debug=DEBUG,
)

run_result

Already processed: 40
Remaining: 8 / 48

Processing 41/48 -> Order d36d882e3c38
----- KB QUERY -----
medication error classification taxonomy relevant drugs parecoxib sodium; ondansetron, ondansetron hydrochloride
----- KB SNIPPETS -----
{"id": "med_error_definition", "type": "definition", "topic": "medication_error_definition", "title": "Medication Error Definition", "text": "A failure in the treatment process that leads to, or has the potential to lead to, harm to the patient.", "keywords": ["medication error", "definition", "harm", "treatment process"]}
{"id": "prescribing_fault_definition", "type": "definition", "topic": "medication_error_definition", "title": "Prescribing Fault", "text": "Failure in decision-making process of prescribing", "keywords": ["prescribing fault", "decision-making", "prescribing process"]}
{"id": "prescription_error_definition", "type": "definition", "topic": "medication_error_definition", "title": "Prescription Error", "text": "Failure in prescription wr

{'results_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/medicheck_results.json',
 'failed_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/failed_cases.json',
 'debug_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/debug_log.json',
 'processed': 48,
 'failed': 0}

In [ ]:
# recheck vector
from openai import OpenAI
import json

client = OpenAI(api_key=OPENAI_API_KEY)

cache = json.load(open(CACHE_PATH))
vs_id = cache["vector_store"]["id"]

vs = client.vector_stores.retrieve(vs_id)

print("status:", vs.status)
print("file_counts:", vs.file_counts)

status: completed
file_counts: FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2)


In [ ]:
import importlib
import run_inference_colab
importlib.reload(run_inference_colab)

from run_inference_colab import run_inference

run_result = run_inference(
    api_key=OPENAI_API_KEY,
    excel_file=EXCEL_FILE,
    group_column=GROUP_COLUMN,
    cache_path=CACHE_PATH,
    output_dir=OUTPUT_DIR,
    model=MODEL,
    sleep_sec=1.0,
    autosave_every=10,
    max_num_results=8,
    debug=DEBUG,
)

run_result

Already processed: 40
Remaining: 8 / 48

Processing 41/48 -> Order d36d882e3c38
----- KB QUERY -----
medication error classification taxonomy relevant drugs parecoxib sodium; ondansetron, ondansetron hydrochloride
----- KB SNIPPETS -----
{"id": "med_error_definition", "type": "definition", "topic": "medication_error_definition", "title": "Medication Error Definition", "text": "A failure in the treatment process that leads to, or has the potential to lead to, harm to the patient.", "keywords": ["medication error", "definition", "harm", "treatment process"]}
{"id": "prescribing_fault_definition", "type": "definition", "topic": "medication_error_definition", "title": "Prescribing Fault", "text": "Failure in decision-making process of prescribing", "keywords": ["prescribing fault", "decision-making", "prescribing process"]}
{"id": "prescription_error_definition", "type": "definition", "topic": "medication_error_definition", "title": "Prescription Error", "text": "Failure in prescription wr

{'results_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/medicheck_results.json',
 'failed_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/failed_cases.json',
 'debug_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/debug_log.json',
 'processed': 48,
 'failed': 0}

# Result

In [ ]:
import json
import os

result_file = f"/content/drive/MyDrive/medicheck_output/partial_results.json"
failed_file = f"/content/drive/MyDrive/medicheck_output/failed_cases.json"
debug_file = f"/content/drive/MyDrive/medicheck_output/debug_log.json"

print("result_file exists:", os.path.exists(result_file))
print("failed_file exists:", os.path.exists(failed_file))
print("debug_file exists:", os.path.exists(debug_file))

with open(result_file, "r", encoding="utf-8") as f:
    results = json.load(f)

with open(failed_file, "r", encoding="utf-8") as f:
    failed = json.load(f)

print("success cases =", len(results))
print("failed cases =", len(failed))

result_file exists: True
failed_file exists: True
debug_file exists: True
success cases = 0
failed cases = 30


In [ ]:
import pandas as pd

rows = []
for order_id, item in results.items():
    rows.append({
        "order_id": item.get("order_id", order_id),
        "has_medication_error": item.get("has_medication_error"),
        "error_categories": " | ".join(item.get("error_categories", [])),
        "error_details": " | ".join(item.get("error_details", [])),
        "implicated_drugs": " | ".join(item.get("implicated_drugs", [])),
    })

df_result = pd.DataFrame(rows)
df_result.head()

""


In [ ]:
import json

with open(f"/content/drive/MyDrive/medicheck_output/debug_log.json", "r") as f:
    logs = json.load(f)

logs[0]

{'order_id': '0280d683793d',
 'error_type': 'ValueError',
 'error': 'Model returned empty text output.',
 'prompt_preview': '\nReview the following prescription order.\n\nOrder ID: 0280d683793d\n\nPrescription data:\n[\n  {\n    "ID": "0280d683793d",\n    "Drug": "diazepam",\n    "Dose": "1 Capsule",\n    "Frequency": "2 Time Daily Morning and Evening"\n  },\n  {\n    "ID": "0280d683793d",\n    "Drug": "ondansetron",\n    "Dose": "1 Tablet",\n    "Frequency": "3 Times Daily Before Breakfast Lunch and "\n  },\n  {\n    "ID": "0280d683793d",\n    "Drug": "hyoscine-n-butyl bromide, hyoscine-n-",\n    "Dose": "1 Tablet",\n    "Frequency": "Every 6 Hours If Needed For Abdominal "\n  },\n  {\n    "ID": "0280d683793d",\n    "Drug": "-",\n    "Dose": "- -",\n    "Frequency": "Every 2-3 Hours When have Symptom"\n  }\n]\n\nTasks:\n1. Decide whether this prescription contains a medication error.\n2. Apply multi-label classification if more than one issue is present.\n3. Classify the issue using t

# inspect failures

In [ ]:
import json

with open(f"{OUTPUT_DIR}/failed_cases.json", "r", encoding="utf-8") as f:
    failed = json.load(f)

print("failed count =", len(failed))
failed[:3]

failed count = 0


[]